# OOD Evaluation — Adaptive Residual Steering

Tests trained SteerNet on **5 out-of-distribution benchmarks** to verify
steering generalizes beyond GSM8K training data.

| Dataset | Type | Tests |
|---|---|---|
| SVAMP | Numeric | Near-domain math robustness |
| GSM-Plus | Numeric | Perturbation resistance (not memorizing) |
| MATH-500 | Numeric | Hard math generalization |
| ARC-Challenge | MCQ | Cross-domain (science reasoning) |
| AQUA-RAT | MCQ | Algebraic reasoning |

**Required artifacts from Phase 1 training run:**
- `checkpoints/steernet.pt`
- `norms_calibration.json`
- `deployment_contract.json`

## 1. Environment Setup

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!nvidia-smi
!pip install -q 'transformers>=4.36.0' 'datasets>=2.16.0' 'accelerate>=0.25.0' matplotlib pandas scipy tqdm

## 2. Clone Repository

In [ ]:
import os, sys

REPO_DIR = '/kaggle/working/adaptive-residual-steering'
BRANCH = 'development'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/Abdullah182155/adaptive-residual-steering.git {REPO_DIR}
else:
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## 3. Upload Artifacts from Phase 1 Run

Upload `steernet.pt`, `norms_calibration.json`, `deployment_contract.json` from your training run.

In [ ]:
import shutil, os

# Point to the kaggle_ars_runner output (same Kaggle session or saved as dataset)
# Option A: From the training notebook output in the same session
ARTIFACT_DIR = '/kaggle/working/adaptive-residual-steering'

# Option B: From a saved Kaggle dataset
# ARTIFACT_DIR = '/kaggle/input/ars-artifacts'

# Check if artifacts exist directly (same-session) or need copying
steernet_path = os.path.join(ARTIFACT_DIR, 'checkpoints/steernet.pt')
steernet_dir = os.path.join(ARTIFACT_DIR, 'checkpoints/steernet')
norms_path = os.path.join(ARTIFACT_DIR, 'norms_calibration.json')
contract_path = os.path.join(ARTIFACT_DIR, 'deployment_contract.json')

print(f'Artifact source: {ARTIFACT_DIR}')
print(f'  steernet.pt:            {"FOUND" if os.path.exists(steernet_path) else "missing"}')
print(f'  steernet/ (dir):        {"FOUND" if os.path.isdir(steernet_dir) else "missing"}')
print(f'  norms_calibration.json: {"FOUND" if os.path.exists(norms_path) else "missing"}')
print(f'  deployment_contract:    {"FOUND" if os.path.exists(contract_path) else "missing"}')

has_steernet = os.path.exists(steernet_path) or os.path.isdir(steernet_dir)
assert has_steernet, 'steernet checkpoint not found!'
assert os.path.exists(norms_path), 'norms_calibration.json not found!'
print('\nAll artifacts ready.')

## 4. OOD Evaluation Config

Control sample count per dataset here.

In [ ]:
# ============================================================
# OOD EVALUATION CONFIG — adjust sample counts here
# ============================================================
N_SVAMP     = 100
N_GSM_PLUS  = 100
N_MATH500   = 100
N_ARC       = 100
N_AQUA_RAT  = 100

MAX_NEW_TOKENS = 256    # generation length

DATASETS_CONFIG = {
    'svamp':    N_SVAMP,
    'gsm_plus': N_GSM_PLUS,
    'math500':  N_MATH500,
    'arc':      N_ARC,
    'aqua_rat': N_AQUA_RAT,
}

# Remove any dataset with N=0
DATASETS_CONFIG = {k: v for k, v in DATASETS_CONFIG.items() if v > 0}

total = sum(DATASETS_CONFIG.values())
print(f'OOD Evaluation: {len(DATASETS_CONFIG)} datasets, {total} total samples')
for name, n in DATASETS_CONFIG.items():
    print(f'  {name}: {n} samples')

## 5. Load Model + SteerNet

In [ ]:
import torch, os, zipfile
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.configs.config import RSCConfig
from src.models.wrapper import Phi2WithRSC

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

cfg = RSCConfig()
cfg.layer_norms_calibration = os.path.join(ARTIFACT_DIR, 'norms_calibration.json')

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

backbone = AutoModelForCausalLM.from_pretrained(
    cfg.model_name, torch_dtype=torch.float32, trust_remote_code=True
).to(device)

model = Phi2WithRSC(backbone, cfg, device)

def _load_checkpoint(base_dir):
    """Load SteerNet checkpoint from .pt file or extracted directory."""
    pt_path = os.path.join(base_dir, 'checkpoints/steernet.pt')
    dir_path = os.path.join(base_dir, 'checkpoints/steernet')
    if os.path.isfile(pt_path):
        return torch.load(pt_path, map_location=device, weights_only=False)
    if os.path.isdir(dir_path):
        repack = os.path.join(base_dir, 'checkpoints/steernet_repacked.pt')
        print(f'Repackaging directory checkpoint -> {repack}')
        with zipfile.ZipFile(repack, 'w') as zf:
            for root, dirs, files in os.walk(dir_path):
                for f in files:
                    if f.startswith('.'):
                        continue
                    full = os.path.join(root, f)
                    rel = os.path.relpath(full, dir_path)
                    zf.write(full, 'archive/' + rel.replace(os.sep, '/'))
        return torch.load(repack, map_location=device, weights_only=False)
    raise FileNotFoundError(f'No steernet checkpoint in {base_dir}/')

ckpt = _load_checkpoint(ARTIFACT_DIR)
steer_dict = ckpt.get('steer_nets', ckpt)
missing, unexpected = model.steer_nets.load_state_dict(steer_dict, strict=False)
print(f'Loaded SteerNet: {len(steer_dict)} tensors, missing={len(missing)}, unexpected={len(unexpected)}')
if ckpt_meta := ckpt.get('metadata'):
    print(f'Checkpoint metadata: {ckpt_meta}')

model.eval()
model.set_telemetry(False)
print('Model ready for OOD evaluation.')

## 6. Run OOD Evaluation

In [ ]:
from src.evaluation.ood_benchmark import run_all_ood

ood_results = run_all_ood(
    model=model,
    tokenizer=tokenizer,
    device=device,
    datasets_config=DATASETS_CONFIG,
    max_new_tokens=MAX_NEW_TOKENS,
    fixed_layers=cfg.fixed_layers,
    report_dir='report',
)

## 7. Detailed Results

In [ ]:
import json

for dname, res in ood_results.items():
    print(f"\n{'='*50}")
    print(f"{dname.upper()} (n={res['n']}, type={res['answer_type']})")
    print(f"{'='*50}")
    b = res.get('baseline', {})
    f = res.get('fixed', {})
    print(f"  Baseline:  {b.get('accuracy', 0):.1f}%")
    print(f"  Steered:   {f.get('accuracy', 0):.1f}%")
    if 'paired' in res:
        p = res['paired']
        print(f"  Delta:     {p['delta_pp']:+.1f} pp")
        print(f"  95% CI:    [{p['ci_95'][0]:.1f}, {p['ci_95'][1]:.1f}]")
        print(f"  Wins/Losses: {p['wins']}/{p['losses']}")
        print(f"  McNemar p: {p['mcnemar_p']:.4f}")

print(f"\n\nFull results JSON:")
print(json.dumps(ood_results, indent=2, default=str))

## 8. Save Artifacts

In [ ]:
import shutil, os

out_dir = '/kaggle/working/ood_artifacts'
os.makedirs(out_dir, exist_ok=True)

if os.path.exists('report/ood'):
    shutil.copytree('report/ood', f'{out_dir}/ood', dirs_exist_ok=True)
    print('Copied: report/ood/')

print(f'\nArtifacts saved to {out_dir}:')
for root, dirs, files in os.walk(out_dir):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f'  {os.path.relpath(path, out_dir)} ({size/1024:.1f} KB)')